In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent

import pandas as pd

import json
from src import constants as Con
from src.data_paths import (
    GATHERERS_FOLDS_DIR,
    GATHERERS_PROCESSED_PATH,
    HUNTERS_FOLDS_DIR,
    HUNTERS_PROCESSED_PATH,
    ALL_PARTICIPANTS_PROCESSED_PATH,

    COL_SAVE_PATH,

    DATA_DIR,
)


import predictive_modeling.answer_correctness.feature_groups as FG

from src.predictive_modeling.answer_correctness.run_model_bundles import (
    run_full_features_correctness_bundle,
    run_full_features_correctness_julia_glmer_bundle,
    run_full_features_correctness_julia_glmer_fit_all,

    build_train_test_trial_dfs
)
from predictive_modeling.answer_correctness.run_report_analysis import collect_and_plot_correctness_runs

from predictive_modeling.common.feature_selection import (
    correlation_prune_features,
    aic_forward_select_logit
)
from predictive_modeling.answer_correctness.model_data import (
    build_trial_level_model_df,
    save_all_features,
    load_all_features,
)
from predictive_modeling.answer_correctness.models.logreg_model import TrialLevelLogRegModel
from predictive_modeling.answer_correctness.cross_validation import run_cross_validation_on_predefined_folds
from predictive_modeling.answer_correctness.cross_validation import show_cv_results, plot_cv_metric_by_regime_pretty
from predictive_modeling.answer_correctness.participant_level import evaluate_logreg_on_answer_correctness_leave_one_trial_out

from predictive_modeling.answer_correctness.evaluation_core import evaluate_models_on_prepared_split

from predictive_modeling.answer_correctness.generate_column_options import generate_all_feature_column_sets
from predictive_modeling.answer_correctness.generate_column_options import run_correctness_bundle_for_saved_column_sets

from predictive_modeling.answer_correctness.generate_column_options import \
    generate_k_most_frequent_feature_sets_from_full_files
from predictive_modeling.answer_correctness.generate_column_options import plot_feature_frequency_from_full_jsons

from predictive_modeling.common.data_utils import vif_from_bundle




## Unlikely Analysis

In [ ]:
trial_df = load_all_features()

In [ ]:
all_participants = pd.read_csv(ALL_PARTICIPANTS_PROCESSED_PATH)

In [ ]:
train_df, test_df, split_info = build_train_test_trial_dfs(
    df=all_participants,
    test_regimes=["both"],    # was split_group_cols=[Con.PARTICIPANT_ID, Con.TRIAL_ID]
    test_split="test",
    keep_cols=[Con.TEXT_ID_WITH_Q_COLUMN],
    random_state=42,
)

In [ ]:
path = COL_SAVE_PATH / "10_most_frequent_last_all.json"
with open(path, "r", encoding="utf-8") as f:
    columns2 = json.load(f)["columns"]

In [ ]:
results = evaluate_models_on_prepared_split(
    models=[TrialLevelLogRegModel()],
    train_df=train_df,
    test_df=test_df,
    target_col=Con.IS_CORRECT_COLUMN,
    feature_cols=columns2,
)

res = results["trial_level_log_reg"]

In [ ]:
from predictive_modeling.answer_correctness.unlikely_analysis import summarize_unlikely_items_with_viz

out = summarize_unlikely_items_with_viz(
    res,
    high_thresh=0.80,
    low_thresh=0.20,
    top_n_groups=15,
)

# mean of "HP-W" - "LP-W"
# mean of "LP-R" - "HP-R"

In [ ]:
from predictive_modeling.answer_correctness.unlikely_analysis import \
    summarize_feature_means_for_probability_outcome_groups

feature_group_means_df = summarize_feature_means_for_probability_outcome_groups(
    result=res,
    feature_cols=columns2,
    high_thresh=0.80,
    low_thresh=0.20,
)

display(feature_group_means_df)

In [ ]:
from predictive_modeling.answer_correctness.unlikely_analysis import plot_feature_group_separation

plot_feature_group_separation(
    feature_group_means_df=feature_group_means_df,
    top_n=20,
)

#normalize